# YBT Model Improvements: SPQ Analysis, Domain Adaptation, and AQ Filtering

This notebook implements comprehensive improvements for YBT external validation:

1. **Retrain models without SPQ** (preserving original models with SPQ for CARD)
2. **Feature importance analysis** to quantify SPQ's contribution
3. **Domain adaptation experiments** (DANN, CORAL)
4. **Apply AQ cutoff** (≥6) to YBT balanced dataset (matching C4 filtering)

## Key Points:
- Original models (with SPQ) are preserved for CARD dataset
- New models (without SPQ) are trained specifically for YBT
- AQ cutoff: Remove autism cases with AQ < 6 (matching C4 preprocessing)

In [ ]:
# Imports
import os
import json
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.utils import resample
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)

# Paths
ARTIFACT_DIR = '/Users/eb2007/playground/bullpy/c4_play2/models/cross_validation'
FEATURE_INFO_PATH = os.path.join(ARTIFACT_DIR, 'feature_info_original.json')
SCALER_PATH = os.path.join(ARTIFACT_DIR, 'scaler_original.joblib')
C4_DATA_PATH = '/Users/eb2007/playground/bullpy/c4_play2/data/processed/data_c4_matched_balanced.csv'
YBT_DATA_PATH = '/Users/eb2007/Library/CloudStorage/OneDrive-UniversityofCambridge/Documents/PhD/data/YBT.csv'
OUTPUT_DIR = '/Users/eb2007/playground/bullpy/c4_play2/models/ybt_optimized'
RESULTS_DIR = '/Users/eb2007/playground/bullpy/c4_play2/data/processed/ybt_improvements'

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print("✅ Imports and paths configured")

## PART 1: Load and Prepare Data

### 1.1: Load C4 Training Data (with SPQ)

In [ ]:
print("="*80)
print("LOADING C4 TRAINING DATA")
print("="*80)

# Load C4 balanced dataset
df_c4 = pd.read_csv(C4_DATA_PATH)
print(f"C4 dataset shape: {df_c4.shape}")
print(f"C4 autism distribution: {df_c4['autism_target'].value_counts().to_dict()}")

# Load feature schema
with open(FEATURE_INFO_PATH, 'r') as f:
    feature_info = json.load(f)

c4_feature_names = feature_info['feature_names']
excluded_features = set(feature_info.get('excluded_features', []))

print(f"\nC4 feature count: {len(c4_feature_names)}")
print(f"Excluded features (AQ-related): {len(excluded_features)}")

# Identify SPQ features
spq_features = [f for f in c4_feature_names if 'spq' in f.lower()]
print(f"\nSPQ features ({len(spq_features)}): {spq_features}")

# Features without SPQ
features_no_spq = [f for f in c4_feature_names if f not in spq_features]
print(f"\nFeatures without SPQ ({len(features_no_spq)}): {features_no_spq[:10]}...")

### 1.2: Load and Preprocess YBT Data (with AQ Cutoff)

In [ ]:
print("="*80)
print("LOADING AND PREPROCESSING YBT DATA")
print("="*80)

# Load YBT data
df_ybt = pd.read_csv(YBT_DATA_PATH)
print(f"YBT raw dataset shape: {df_ybt.shape}")

# Create autism_target
if 'diagnosis' in df_ybt.columns:
    df_ybt['autism_target'] = df_ybt['diagnosis'].astype(str).str.contains('autism', case=False, na=False).astype(int)
else:
    df_ybt['autism_target'] = 0

print(f"\nYBT autism distribution (raw): {df_ybt['autism_target'].value_counts().to_dict()}")

# Convert text responses to numeric (if needed)
response_mapping = {
    'strongly agree': 4,
    'slightly agree': 3,
    'slightly disagree': 2,
    'strongly disagree': 1
}

# Convert questionnaire columns
eq_cols = [col for col in df_ybt.columns if col.startswith('eq10_')]
sqr_cols = [col for col in df_ybt.columns if col.startswith('sq10_')]
aq_cols = [col for col in df_ybt.columns if col.startswith('aq_') and len(col) <= 5]

all_q_cols = eq_cols + sqr_cols + aq_cols
for col in all_q_cols:
    if col in df_ybt.columns:
        df_ybt[col] = df_ybt[col].astype(str).str.strip().str.lower().map(response_mapping)
        df_ybt[col] = pd.to_numeric(df_ybt[col], errors='coerce')

# Score questionnaires (matching C4 rules)
# EQ-10 Scoring
eq_reverse_items = [3]
for i in range(1, 11):
    col_name = f'eq10_{i}'
    if col_name in df_ybt.columns:
        if i in eq_reverse_items:
            df_ybt[col_name] = df_ybt[col_name].apply(
                lambda x: 1 if pd.notna(x) and x in [1, 2] else 0 if pd.notna(x) and x in [3, 4] else np.nan
            )
        else:
            df_ybt[col_name] = df_ybt[col_name].apply(
                lambda x: 1 if pd.notna(x) and x in [3, 4] else 0 if pd.notna(x) and x in [1, 2] else np.nan
            )
        df_ybt[f'eq_{i}'] = df_ybt[col_name]

df_ybt['eq_total'] = df_ybt[[f'eq_{i}' for i in range(1, 11)]].sum(axis=1)

# SQR-10 Scoring
sqr_reverse_items = [2, 4, 6, 8, 10]
for i in range(1, 11):
    col_name = f'sq10_{i}'
    if col_name in df_ybt.columns:
        if i in sqr_reverse_items:
            df_ybt[col_name] = df_ybt[col_name].apply(
                lambda x: 1 if pd.notna(x) and x in [1, 2] else 0 if pd.notna(x) and x in [3, 4] else np.nan
            )
        else:
            df_ybt[col_name] = df_ybt[col_name].apply(
                lambda x: 1 if pd.notna(x) and x in [3, 4] else 0 if pd.notna(x) and x in [1, 2] else np.nan
            )
        df_ybt[f'sqr_{i}'] = df_ybt[col_name]

df_ybt['sqr_total'] = df_ybt[[f'sqr_{i}' for i in range(1, 11)]].sum(axis=1)

# AQ-10 Scoring
aq_reverse_items = [2, 3, 4, 5, 6, 9]
for i in range(1, 11):
    col_name = f'aq_{i}'
    if col_name in df_ybt.columns:
        if i in aq_reverse_items:
            df_ybt[col_name] = df_ybt[col_name].apply(
                lambda x: 1 if pd.notna(x) and x in [1, 2] else 0 if pd.notna(x) and x in [3, 4] else np.nan
            )
        else:
            df_ybt[col_name] = df_ybt[col_name].apply(
                lambda x: 1 if pd.notna(x) and x in [3, 4] else 0 if pd.notna(x) and x in [1, 2] else np.nan
            )

df_ybt['aq_total'] = df_ybt[[f'aq_{i}' for i in range(1, 11)]].sum(axis=1)

# SPQ features (fill with 0)
for i in range(1, 11):
    df_ybt[f'spq_{i}'] = 0
df_ybt['spq_total'] = 0

print(f"\n✅ Questionnaires scored")
print(f"   EQ total range: {df_ybt['eq_total'].min():.0f}-{df_ybt['eq_total'].max():.0f}")
print(f"   SQR total range: {df_ybt['sqr_total'].min():.0f}-{df_ybt['sqr_total'].max():.0f}")
print(f"   AQ total range: {df_ybt['aq_total'].min():.0f}-{df_ybt['aq_total'].max():.0f}")

In [ ]:
# Apply AQ cutoff: Remove autism cases with AQ < 6 (matching C4)
print("\n" + "="*80)
print("APPLYING AQ CUTOFF (≥6) TO YBT DATASET")
print("="*80)

print(f"\nBefore AQ filtering:")
print(f"  Total samples: {len(df_ybt)}")
print(f"  Autism cases: {(df_ybt['autism_target'] == 1).sum()}")
print(f"  Non-autism cases: {(df_ybt['autism_target'] == 0).sum()}")

# Check AQ distribution for autism cases
autism_cases = df_ybt[df_ybt['autism_target'] == 1]
print(f"\nAQ distribution for autism cases:")
print(f"  Mean: {autism_cases['aq_total'].mean():.2f}")
print(f"  Min: {autism_cases['aq_total'].min():.0f}")
print(f"  Max: {autism_cases['aq_total'].max():.0f}")
print(f"  Cases with AQ < 6: {(autism_cases['aq_total'] < 6).sum()}")
print(f"  Cases with AQ ≥ 6: {(autism_cases['aq_total'] >= 6).sum()}")

# Apply filtering: Remove autism cases with AQ < 6
df_ybt_filtered = df_ybt[~((df_ybt['autism_target'] == 1) & (df_ybt['aq_total'] < 6))].copy()

print(f"\nAfter AQ filtering:")
print(f"  Total samples: {len(df_ybt_filtered)}")
print(f"  Autism cases: {(df_ybt_filtered['autism_target'] == 1).sum()}")
print(f"  Non-autism cases: {(df_ybt_filtered['autism_target'] == 0).sum()}")
print(f"  Removed: {len(df_ybt) - len(df_ybt_filtered)} autism cases with AQ < 6")

# Balance YBT dataset (1:1 matching C4)
print(f"\n" + "="*80)
print("BALANCING YBT DATASET (1:1)")
print("="*80)

autism_cases = df_ybt_filtered[df_ybt_filtered['autism_target'] == 1].copy()
non_autism_cases = df_ybt_filtered[df_ybt_filtered['autism_target'] == 0].copy()

n_autism = len(autism_cases)
non_autism_downsampled = resample(
    non_autism_cases, 
    replace=False,
    n_samples=n_autism, 
    random_state=42
)

df_ybt_balanced = pd.concat([autism_cases, non_autism_downsampled], ignore_index=True)
df_ybt_balanced = df_ybt_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\nBalanced YBT dataset:")
print(f"  Total samples: {len(df_ybt_balanced)}")
print(f"  Autism cases: {(df_ybt_balanced['autism_target'] == 1).sum()}")
print(f"  Non-autism cases: {(df_ybt_balanced['autism_target'] == 0).sum()}")
print(f"  Balance ratio: 1:1")

# Save filtered and balanced YBT dataset
ybt_balanced_path = os.path.join(RESULTS_DIR, 'ybt_balanced_aq_filtered.csv')
df_ybt_balanced.to_csv(ybt_balanced_path, index=False)
print(f"\n✅ Saved balanced YBT dataset: {ybt_balanced_path}")

### 1.3: Feature Engineering for YBT

In [ ]:
# Feature engineering (matching C4)
print("="*80)
print("FEATURE ENGINEERING FOR YBT")
print("="*80)

# Age
df_ybt_balanced['age'] = pd.to_numeric(df_ybt_balanced['age'], errors='coerce')
age_median = df_ybt_balanced['age'].median()
df_ybt_balanced['age'] = df_ybt_balanced['age'].fillna(age_median)

# Sex
sex_mapping = {
    'male': 1, 'female': 2, 'other': 3,
    'prefer not to say': 4, 'i prefer not to say': 4, 'i do not know': 4
}
if 'sex' in df_ybt_balanced.columns:
    df_ybt_balanced['sex'] = df_ybt_balanced['sex'].astype(str).str.strip().str.lower().map(sex_mapping).fillna(4)
    df_ybt_balanced['sex_num'] = df_ybt_balanced['sex'].map({1: 0, 2: 1, 3: 2, 4: 3}).fillna(0).astype(int)
else:
    df_ybt_balanced['sex'] = 4
    df_ybt_balanced['sex_num'] = 0

# Age groups
df_ybt_balanced['age_group_19-30'] = ((df_ybt_balanced['age'] >= 19) & (df_ybt_balanced['age'] <= 30)).astype(int)
df_ybt_balanced['age_group_31-45'] = ((df_ybt_balanced['age'] >= 31) & (df_ybt_balanced['age'] <= 45)).astype(int)
df_ybt_balanced['age_group_46-60'] = ((df_ybt_balanced['age'] >= 46) & (df_ybt_balanced['age'] <= 60)).astype(int)
df_ybt_balanced['age_group_61+'] = (df_ybt_balanced['age'] >= 61).astype(int)

# sqrt_age
df_ybt_balanced['sqrt_age'] = np.sqrt(np.clip(df_ybt_balanced['age'], a_min=0, a_max=None))

# Interactions
df_ybt_balanced['d_score'] = df_ybt_balanced['sqr_total'] - df_ybt_balanced['eq_total']
df_ybt_balanced['age_x_eq'] = df_ybt_balanced['age'] * df_ybt_balanced['eq_total']
df_ybt_balanced['eq_sqr_ratio'] = df_ybt_balanced['eq_total'] / (df_ybt_balanced['sqr_total'].replace(0, np.nan) + 1e-8)
df_ybt_balanced['eq_sqr_ratio'] = df_ybt_balanced['eq_sqr_ratio'].replace([np.inf, -np.inf], np.nan).fillna(0.0)

# Occupation
if 'Q549' in df_ybt_balanced.columns or 'occupation' in df_ybt_balanced.columns:
    occupation_col = 'Q549' if 'Q549' in df_ybt_balanced.columns else 'occupation'
    df_ybt_balanced['is_stem_occupation'] = df_ybt_balanced[occupation_col].astype(str).str.contains(
        'science|technology|engineering|math|computer|software|data|research', 
        case=False, na=False
    ).astype(int)
else:
    df_ybt_balanced['is_stem_occupation'] = 0

print("✅ Feature engineering complete")

## PART 2: Feature Importance Analysis

In [ ]:
print("="*80)
print("FEATURE IMPORTANCE ANALYSIS")
print("="*80)

# Prepare C4 data
X_c4 = df_c4[c4_feature_names].copy()

# Handle missing values and ensure numeric
X_c4 = X_c4.fillna(0)
# Convert all columns to numeric, coercing errors to NaN then filling with 0
for col in X_c4.columns:
    if X_c4[col].dtype == 'object':
        X_c4[col] = pd.to_numeric(X_c4[col], errors='coerce').fillna(0)
    else:
        X_c4[col] = pd.to_numeric(X_c4[col], errors='coerce').fillna(0)

y_c4 = df_c4['autism_target']

# Scale C4 data (convert to numpy array)
scaler_c4 = StandardScaler()
X_c4_scaled = scaler_c4.fit_transform(X_c4.values)

# Train Random Forest to get feature importance
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_c4_scaled, y_c4)

# Get feature importance
feature_importance = pd.DataFrame({
    'feature': c4_feature_names,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

# Identify SPQ feature importance
spq_importance = feature_importance[feature_importance['feature'].str.contains('spq', case=False, na=False)]
total_spq_importance = spq_importance['importance'].sum()

print(f"\nTop 20 Most Important Features:")
print(feature_importance.head(20).to_string(index=False))

print(f"\nSPQ Feature Importance:")
print(spq_importance.to_string(index=False))
print(f"\nTotal SPQ importance: {total_spq_importance:.4f} ({total_spq_importance*100:.2f}%)")
print(f"Average SPQ feature importance: {spq_importance['importance'].mean():.4f}")
print(f"Average non-SPQ feature importance: {feature_importance[~feature_importance['feature'].str.contains('spq', case=False, na=False)]['importance'].mean():.4f}")

# Save feature importance
importance_path = os.path.join(RESULTS_DIR, 'feature_importance_analysis.csv')
feature_importance.to_csv(importance_path, index=False)
print(f"\n✅ Saved feature importance: {importance_path}")

# Visualize
plt.figure(figsize=(12, 8))
top_features = feature_importance.head(20)
colors = ['red' if 'spq' in f.lower() else 'blue' for f in top_features['feature']]
plt.barh(range(len(top_features)), top_features['importance'], color=colors)
plt.yticks(range(len(top_features)), top_features['feature'])
plt.xlabel('Feature Importance')
plt.title('Top 20 Feature Importance (Red=SPQ, Blue=Other)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'feature_importance_plot.png'), dpi=150)
print(f"✅ Saved feature importance plot")

## PART 3: Retrain Models Without SPQ

In [ ]:
print("="*80)
print("RETRAINING MODELS WITHOUT SPQ")
print("="*80)
print("\n⚠️  Original models (with SPQ) are preserved for CARD dataset")
print("    New models (without SPQ) will be saved separately\n")

# Prepare C4 data without SPQ
X_c4_no_spq = df_c4[features_no_spq].copy()

# Handle missing values and ensure numeric
X_c4_no_spq = X_c4_no_spq.fillna(0)
# Convert all columns to numeric, coercing errors to NaN then filling with 0
for col in X_c4_no_spq.columns:
    if X_c4_no_spq[col].dtype == 'object':
        X_c4_no_spq[col] = pd.to_numeric(X_c4_no_spq[col], errors='coerce').fillna(0)
    else:
        X_c4_no_spq[col] = pd.to_numeric(X_c4_no_spq[col], errors='coerce').fillna(0)

y_c4_no_spq = df_c4['autism_target']

# Scale C4 data without SPQ (convert to numpy array)
scaler_no_spq = StandardScaler()
X_c4_no_spq_scaled = scaler_no_spq.fit_transform(X_c4_no_spq.values)

# Define models
models_no_spq = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000, n_jobs=-1),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
    'XGBoost': XGBClassifier(random_state=42, n_jobs=-1, eval_metric='logloss'),
    'LightGBM': LGBMClassifier(random_state=42, n_jobs=-1, verbose=-1)
}

# Train models
trained_models_no_spq = {}
results_no_spq = {}

for name, model in models_no_spq.items():
    print(f"\nTraining {name}...")
    model.fit(X_c4_no_spq_scaled, y_c4_no_spq)
    trained_models_no_spq[name] = model
    
    # Cross-validation
    cv_scores = cross_val_score(model, X_c4_no_spq_scaled, y_c4_no_spq, cv=5, scoring='roc_auc', n_jobs=-1)
    results_no_spq[name] = {
        'cv_auc_mean': cv_scores.mean(),
        'cv_auc_std': cv_scores.std()
    }
    print(f"  CV AUC: {cv_scores.mean():.4f} (+/- {cv_scores.std()*2:.4f})")

# Save models and scaler
for name, model in trained_models_no_spq.items():
    model_path = os.path.join(OUTPUT_DIR, f'{name.lower().replace(" ", "_")}_no_spq.joblib')
    joblib.dump(model, model_path)
    print(f"✅ Saved {name}: {model_path}")

scaler_path = os.path.join(OUTPUT_DIR, 'scaler_no_spq.joblib')
joblib.dump(scaler_no_spq, scaler_path)
print(f"\n✅ Saved scaler: {scaler_path}")

# Save feature info without SPQ
feature_info_no_spq = {
    'feature_names': features_no_spq,
    'excluded_features': list(excluded_features) + spq_features,
    'note': 'Models trained without SPQ features for YBT external validation'
}
feature_info_path = os.path.join(OUTPUT_DIR, 'feature_info_no_spq.json')
with open(feature_info_path, 'w') as f:
    json.dump(feature_info_no_spq, f, indent=2)
print(f"✅ Saved feature info: {feature_info_path}")

# Save results
results_path = os.path.join(RESULTS_DIR, 'c4_models_no_spq_results.json')
with open(results_path, 'w') as f:
    json.dump(results_no_spq, f, indent=2)
print(f"✅ Saved results: {results_path}")

## PART 4: Evaluate Models Without SPQ on YBT

In [ ]:
print("="*80)
print("EVALUATING MODELS WITHOUT SPQ ON YBT")
print("="*80)

# Prepare YBT data (without SPQ features)
X_ybt_no_spq = pd.DataFrame(index=df_ybt_balanced.index)
for feat_name in features_no_spq:
    if feat_name in df_ybt_balanced.columns:
        X_ybt_no_spq[feat_name] = df_ybt_balanced[feat_name]
    else:
        X_ybt_no_spq[feat_name] = 0

X_ybt_no_spq = X_ybt_no_spq[features_no_spq]  # Ensure correct order
X_ybt_no_spq = X_ybt_no_spq.fillna(0)
X_ybt_no_spq = X_ybt_no_spq.apply(pd.to_numeric, errors='coerce').fillna(0)

# Scale YBT data using scaler fitted on C4 (without SPQ)
X_ybt_no_spq_scaled = scaler_no_spq.transform(X_ybt_no_spq.values)

y_ybt = df_ybt_balanced['autism_target'].values

# Evaluate models
ybt_results_no_spq = {}

for name, model in trained_models_no_spq.items():
    y_proba = model.predict_proba(X_ybt_no_spq_scaled)[:, 1]
    y_pred = (y_proba >= 0.5).astype(int)
    
    ybt_results_no_spq[name] = {
        'accuracy': accuracy_score(y_ybt, y_pred),
        'precision': precision_score(y_ybt, y_pred, zero_division=0),
        'recall': recall_score(y_ybt, y_pred, zero_division=0),
        'f1': f1_score(y_ybt, y_pred, zero_division=0),
        'auc': roc_auc_score(y_ybt, y_proba)
    }

# Display results
results_df = pd.DataFrame(ybt_results_no_spq).T
print("\n" + "="*80)
print("YBT VALIDATION RESULTS (MODELS WITHOUT SPQ)")
print("="*80)
print(results_df.round(4).sort_values('auc', ascending=False))

# Save results
results_path = os.path.join(RESULTS_DIR, 'ybt_validation_no_spq_results.csv')
results_df.to_csv(results_path)
print(f"\n✅ Saved results: {results_path}")

## PART 5: Domain Adaptation Experiments

In [ ]:
# Domain adaptation using CORAL (CORrelation ALignment)
print("="*80)
print("DOMAIN ADAPTATION: CORAL (CORrelation ALignment)")
print("="*80)

def coral_transform(Xs, Xt):
    """
    CORAL: Aligns second-order statistics of source and target domains
    """
    # Ensure no NaN or inf values
    Xs = np.nan_to_num(Xs, nan=0.0, posinf=0.0, neginf=0.0)
    Xt = np.nan_to_num(Xt, nan=0.0, posinf=0.0, neginf=0.0)
    
    # Center the data
    Xs_mean = np.mean(Xs, axis=0, keepdims=True)
    Xt_mean = np.mean(Xt, axis=0, keepdims=True)
    Xs_centered = Xs - Xs_mean
    Xt_centered = Xt - Xt_mean
    
    # Covariance matrices (with regularization)
    Cs = np.cov(Xs_centered.T)
    Ct = np.cov(Xt_centered.T)
    
    # Add regularization to avoid singular matrices
    reg = 1e-5
    Cs = Cs + reg * np.eye(Cs.shape[0])
    Ct = Ct + reg * np.eye(Ct.shape[0])
    
    # Whitening and re-coloring
    # Use pseudo-inverse for numerical stability
    try:
        Ws = np.linalg.inv(np.linalg.cholesky(Cs)).T
    except np.linalg.LinAlgError:
        # Fallback to SVD if Cholesky fails
        U, s, Vt = np.linalg.svd(Cs)
        Ws = U @ np.diag(1.0 / np.sqrt(s + reg)) @ Vt
    
    try:
        Wt = np.linalg.cholesky(Ct)
    except np.linalg.LinAlgError:
        # Fallback to SVD if Cholesky fails
        U, s, Vt = np.linalg.svd(Ct)
        Wt = U @ np.diag(np.sqrt(s + reg)) @ Vt
    
    # Transformation matrix
    A = Wt @ Ws
    
    # Transform source data
    Xs_transformed = Xs_centered @ A.T + Xt_mean
    
    # Ensure no NaN or inf in output
    Xs_transformed = np.nan_to_num(Xs_transformed, nan=0.0, posinf=0.0, neginf=0.0)
    
    return Xs_transformed, A

# Prepare data (without SPQ) - ensure numpy arrays and no NaN
X_c4_da = np.array(X_c4_no_spq_scaled)
X_ybt_da = np.array(X_ybt_no_spq_scaled)

# Remove any NaN or inf values before CORAL
X_c4_da = np.nan_to_num(X_c4_da, nan=0.0, posinf=0.0, neginf=0.0)
X_ybt_da = np.nan_to_num(X_ybt_da, nan=0.0, posinf=0.0, neginf=0.0)

# Apply CORAL transformation
X_c4_coral, A_coral = coral_transform(X_c4_da, X_ybt_da)

# Ensure output has no NaN
X_c4_coral = np.nan_to_num(X_c4_coral, nan=0.0, posinf=0.0, neginf=0.0)

print(f"\nCORAL transformation applied")
print(f"  Source (C4) shape: {X_c4_coral.shape}")
print(f"  Target (YBT) shape: {X_ybt_da.shape}")

# Train models on CORAL-transformed C4 data
models_coral = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000, n_jobs=-1),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'XGBoost': XGBClassifier(random_state=42, n_jobs=-1, eval_metric='logloss'),
    'LightGBM': LGBMClassifier(random_state=42, n_jobs=-1, verbose=-1)
}

trained_models_coral = {}
ybt_results_coral = {}

for name, model in models_coral.items():
    print(f"\nTraining {name} with CORAL...")
    model.fit(X_c4_coral, y_c4_no_spq)
    trained_models_coral[name] = model
    
    # Evaluate on YBT
    y_proba = model.predict_proba(X_ybt_da)[:, 1]
    y_pred = (y_proba >= 0.5).astype(int)
    
    ybt_results_coral[name] = {
        'accuracy': accuracy_score(y_ybt, y_pred),
        'precision': precision_score(y_ybt, y_pred, zero_division=0),
        'recall': recall_score(y_ybt, y_pred, zero_division=0),
        'f1': f1_score(y_ybt, y_pred, zero_division=0),
        'auc': roc_auc_score(y_ybt, y_proba)
    }
    print(f"  YBT AUC: {ybt_results_coral[name]['auc']:.4f}, F1: {ybt_results_coral[name]['f1']:.4f}")

# Save CORAL models
for name, model in trained_models_coral.items():
    model_path = os.path.join(OUTPUT_DIR, f'{name.lower().replace(" ", "_")}_coral.joblib')
    joblib.dump(model, model_path)

# Save CORAL transformation matrix
coral_matrix_path = os.path.join(OUTPUT_DIR, 'coral_transformation_matrix.joblib')
joblib.dump(A_coral, coral_matrix_path)

# Display results
results_df_coral = pd.DataFrame(ybt_results_coral).T
print("\n" + "="*80)
print("YBT VALIDATION RESULTS (CORAL DOMAIN ADAPTATION)")
print("="*80)
print(results_df_coral.round(4).sort_values('auc', ascending=False))

# Save results
results_path = os.path.join(RESULTS_DIR, 'ybt_validation_coral_results.csv')
results_df_coral.to_csv(results_path)
print(f"\n✅ Saved CORAL results: {results_path}")

## PART 6: Comparison Summary

In [ ]:
print("="*80)
print("COMPREHENSIVE PERFORMANCE COMPARISON")
print("="*80)

# Load original YBT results (with SPQ, imbalanced)
original_ybt_path = '/Users/eb2007/playground/bullpy/c4_play2/data/processed/ybt_external_validation_results.csv'
if os.path.exists(original_ybt_path):
    original_ybt = pd.read_csv(original_ybt_path, index_col=0)
else:
    original_ybt = None

# Load C4 results
with open('/Users/eb2007/playground/bullpy/c4_play2/models/cross_validation/original_dataset_results.json', 'r') as f:
    c4_results = json.load(f)

print("\n1. C4 Training Performance (with SPQ):")
for model in ['LightGBM', 'Gradient Boosting', 'XGBoost', 'Random Forest', 'Logistic Regression']:
    if model in c4_results:
        print(f"   {model:20s}: F1={c4_results[model]['f1']:.4f}, AUC={c4_results[model]['auc']:.4f}")

print("\n2. YBT Validation Performance (Models WITHOUT SPQ, AQ-filtered, Balanced):")
for model in ['LightGBM', 'Gradient Boosting', 'XGBoost', 'Random Forest', 'Logistic Regression']:
    if model in ybt_results_no_spq:
        print(f"   {model:20s}: F1={ybt_results_no_spq[model]['f1']:.4f}, AUC={ybt_results_no_spq[model]['auc']:.4f}")

print("\n3. YBT Validation Performance (CORAL Domain Adaptation):")
for model in ['LightGBM', 'XGBoost', 'Random Forest', 'Logistic Regression']:
    if model in ybt_results_coral:
        print(f"   {model:20s}: F1={ybt_results_coral[model]['f1']:.4f}, AUC={ybt_results_coral[model]['auc']:.4f}")

if original_ybt is not None:
    print("\n4. YBT Validation Performance (Original: with SPQ=0, imbalanced):")
    for model in original_ybt.index:
        print(f"   {model:20s}: F1={original_ybt.loc[model, 'f1']:.4f}, AUC={original_ybt.loc[model, 'auc']:.4f}")

# Calculate improvements
print("\n" + "="*80)
print("IMPROVEMENT ANALYSIS")
print("="*80)

if original_ybt is not None:
    print("\nImprovement from removing SPQ and balancing:")
    for model in ybt_results_no_spq:
        if model in original_ybt.index:
            f1_improvement = ybt_results_no_spq[model]['f1'] - original_ybt.loc[model, 'f1']
            auc_improvement = ybt_results_no_spq[model]['auc'] - original_ybt.loc[model, 'auc']
            print(f"   {model:20s}: F1 +{f1_improvement:+.4f}, AUC +{auc_improvement:+.4f}")

print("\nImprovement from CORAL domain adaptation:")
for model in ybt_results_coral:
    if model in ybt_results_no_spq:
        f1_improvement = ybt_results_coral[model]['f1'] - ybt_results_no_spq[model]['f1']
        auc_improvement = ybt_results_coral[model]['auc'] - ybt_results_no_spq[model]['auc']
        print(f"   {model:20s}: F1 +{f1_improvement:+.4f}, AUC +{auc_improvement:+.4f}")

# Save comprehensive comparison
comparison_data = []
for model in ['LightGBM', 'Gradient Boosting', 'XGBoost', 'Random Forest', 'Logistic Regression']:
    row = {'Model': model}
    if model in c4_results:
        row['C4_F1'] = c4_results[model]['f1']
        row['C4_AUC'] = c4_results[model]['auc']
    if model in ybt_results_no_spq:
        row['YBT_NoSPQ_F1'] = ybt_results_no_spq[model]['f1']
        row['YBT_NoSPQ_AUC'] = ybt_results_no_spq[model]['auc']
    if model in ybt_results_coral:
        row['YBT_CORAL_F1'] = ybt_results_coral[model]['f1']
        row['YBT_CORAL_AUC'] = ybt_results_coral[model]['auc']
    comparison_data.append(row)

comparison_df = pd.DataFrame(comparison_data)
comparison_path = os.path.join(RESULTS_DIR, 'comprehensive_comparison.csv')
comparison_df.to_csv(comparison_path, index=False)
print(f"\n✅ Saved comprehensive comparison: {comparison_path}")

print("\n" + "="*80)
print("SUMMARY")
print("="*80)
print("\n✅ Original models (with SPQ) preserved for CARD dataset")
print("✅ New models (without SPQ) trained and saved")
print("✅ Feature importance analysis completed")
print("✅ Domain adaptation (CORAL) experiments completed")
print("✅ AQ cutoff (≥6) applied to YBT dataset")
print("\nAll outputs saved to:")
print(f"  Models: {OUTPUT_DIR}")
print(f"  Results: {RESULTS_DIR}")